In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import random

In [ ]:
# Generate or select a subset of data
# For simplicity, let's create some synthetic data or load a subset from Fashion-MNIST
np.random.seed(42)
data = np.random.rand(300, 2)  # Using random data for simplicity (200 points, 2D)

In [ ]:
### 1. Hierarchical Clustering from Scratch ###

def compute_distances(data):
    """Compute pairwise distances between all points."""
    return cdist(data, data, metric='euclidean')

def find_closest_clusters(distances, clusters):
    """Find the two closest clusters based on minimum distance."""
    min_dist = float('inf')
    pair = (None, None)
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            dist = np.min([distances[p1, p2] for p1 in clusters[i] for p2 in clusters[j]])
            if dist < min_dist:
                min_dist = dist
                pair = (i, j)
    return pair

def hierarchical_clustering(data, num_clusters=3):
    distances = compute_distances(data)
    clusters = [[i] for i in range(len(data))]  # Each point starts as its own cluster

    while len(clusters) > num_clusters:
        i, j = find_closest_clusters(distances, clusters)
        clusters[i].extend(clusters[j])  # Merge clusters i and j
        del clusters[j]  # Remove the merged cluster

    labels = np.zeros(len(data))
    for idx, cluster in enumerate(clusters):
        for point in cluster:
            labels[point] = idx
    return labels

labels_hierarchical = hierarchical_clustering(data, num_clusters=3)

In [ ]:
### 2. DBSCAN from Scratch ###

def region_query(data, point_idx, eps):
    """Find all points in the dataset within `eps` distance of `point_idx`."""
    return [i for i, point in enumerate(data) if np.linalg.norm(data[point_idx] - point) <= eps]

def expand_cluster(data, labels, point_idx, cluster_id, eps, min_pts):
    """Expand a new cluster from `point_idx`."""
    seeds = region_query(data, point_idx, eps)
    if len(seeds) < min_pts:
        labels[point_idx] = -1  # Mark as noise
        return False
    else:
        labels[point_idx] = cluster_id
        for seed_idx in seeds:
            labels[seed_idx] = cluster_id

        while seeds:
            current_point = seeds.pop(0)
            result = region_query(data, current_point, eps)
            if len(result) >= min_pts:
                for result_point in result:
                    if labels[result_point] == 0:
                        seeds.append(result_point)
                        labels[result_point] = cluster_id
        return True

def dbscan(data, eps=0.1, min_pts=5):
    labels = np.zeros(len(data))
    cluster_id = 0

    for point_idx in range(len(data)):
        if labels[point_idx] == 0:  # Not visited
            if expand_cluster(data, labels, point_idx, cluster_id + 1, eps, min_pts):
                cluster_id += 1
    return labels

labels_dbscan = dbscan(data, eps=0.1, min_pts=5)

In [ ]:
### 3. Plotting Clusters ###

def plot_clusters(data, labels, title):
    plt.figure(figsize=(8, 6))
    unique_labels = np.unique(labels)
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))
    for k, col in zip(unique_labels, colors):
        class_members = labels == k
        plt.scatter(data[class_members, 0], data[class_members, 1], s=30, color=col, label=f'Cluster {int(k)}')
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend(loc='best')
    plt.show()

# Plot results
plot_clusters(data, labels_hierarchical, "Hierarchical Clustering (Scratch Implementation)")
plot_clusters(data, labels_dbscan, "DBSCAN Clustering (Scratch Implementation)")